# 09 — Final Pipeline: Exploded + Aggregate Movie-Keyword Scores

Produces two output tables from the full upstream stack:

| Output | Rows | Description |
|--------|------|-------------|
| `tmdb_movie_keywords_exploded.csv` | ~2.8M | One row per (movie, keyword) — all lexicon scores |
| `tmdb_movie_scores.csv` | ~1.17M | One row per movie — aggregated scores + metadata |

## Column prefix convention

| Prefix | Source |
|--------|--------|
| `emolex_` | NRC Emotion Lexicon (binary emotion membership) |
| `intensity_` | NRC Emotion Intensity Lexicon (0–1 continuous) |
| `vad_` | NRC VAD Lexicon v2.1 (valence/arousal/dominance, −1 to +1) |
| `scl_` | SCL-OPP / SCL-NMA (phrase composition, opposing polarity) |
| `movie_` | Movie-level aggregate of keyword scores |

**Valence focus:** `vad_valence` is the primary score (NRC VAD). `scl_valence` overrides it
for phrases where SCL-OPP provides a direct measurement. `movie_valence` is the mean across
all scored keywords for the film.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA   = Path('data')
OUTPUT = Path('output')

movies = pd.read_csv(DATA / 'tmdb_movies_clean.csv', low_memory=False)
kw_lex = pd.read_csv(OUTPUT / 'keyword-lexicon/tmdb_keyword_lexicon.csv', low_memory=False)

print(f'Movies:   {len(movies):,}  cols={movies.shape[1]}')
print(f'Keywords: {len(kw_lex):,}  cols={kw_lex.shape[1]}')
print(f'\nkeyword-lexicon columns:')
print(list(kw_lex.columns))

Movies:   1,169,756  cols=26
Keywords: 67,916  cols=46

keyword-lexicon columns:
['keyword', 'n_movies', 'emolex_found', 'intensity_found', 'vad_found', 'emolex_anger', 'emolex_anticipation', 'emolex_disgust', 'emolex_fear', 'emolex_joy', 'emolex_sadness', 'emolex_surprise', 'emolex_trust', 'emolex_positive', 'emolex_negative', 'intensity_anger', 'intensity_anticipation', 'intensity_disgust', 'intensity_fear', 'intensity_joy', 'intensity_sadness', 'intensity_surprise', 'intensity_trust', 'vad_valence', 'vad_arousal', 'vad_dominance', 'in_scl', 'scl_token_coverage', 'valence', 'valence_scale', 'valence_source', 'valence_count', 'valence_variance', 'matched_tokens', 'total_tokens', 'negators_contained', 'modals_contained', 'adverbs_contained', 'has_nma', 'composition_shift', 'abs_composition_shift', 'shift_zscore', 'shift_percentile', 'shift_strong', 'shift_very_strong', 'composition_type']


## Rename keyword lexicon to prefixed convention

In [2]:
# Columns already prefixed: emolex_*, intensity_*, vad_*
# SCL-derived columns need scl_ prefix
# Final computed valence keeps its name but we clarify source

scl_rename = {
    'in_scl':              'scl_found',
    'scl_token_coverage':  'scl_token_coverage',
    'valence':             'keyword_valence',       # best available valence for this keyword
    'valence_scale':       'keyword_valence_scale',
    'valence_source':      'keyword_valence_source',
    'valence_count':       'scl_valence_count',
    'valence_variance':    'scl_valence_variance',
    'matched_tokens':      'scl_matched_tokens',
    'total_tokens':        'scl_total_tokens',
    'negators_contained':  'scl_has_negator',
    'modals_contained':    'scl_has_modal',
    'adverbs_contained':   'scl_has_adverb',
    'has_nma':             'scl_has_nma',
    'composition_shift':   'scl_composition_shift',
    'abs_composition_shift': 'scl_abs_composition_shift',
    'shift_zscore':        'scl_shift_zscore',
    'shift_percentile':    'scl_shift_percentile',
    'shift_strong':        'scl_shift_strong',
    'shift_very_strong':   'scl_shift_very_strong',
    'composition_type':    'scl_composition_type',
}

kw_lex = kw_lex.rename(columns=scl_rename)
print('Renamed columns:')
print(list(kw_lex.columns))

Renamed columns:
['keyword', 'n_movies', 'emolex_found', 'intensity_found', 'vad_found', 'emolex_anger', 'emolex_anticipation', 'emolex_disgust', 'emolex_fear', 'emolex_joy', 'emolex_sadness', 'emolex_surprise', 'emolex_trust', 'emolex_positive', 'emolex_negative', 'intensity_anger', 'intensity_anticipation', 'intensity_disgust', 'intensity_fear', 'intensity_joy', 'intensity_sadness', 'intensity_surprise', 'intensity_trust', 'vad_valence', 'vad_arousal', 'vad_dominance', 'scl_found', 'scl_token_coverage', 'keyword_valence', 'keyword_valence_scale', 'keyword_valence_source', 'scl_valence_count', 'scl_valence_variance', 'scl_matched_tokens', 'scl_total_tokens', 'scl_has_negator', 'scl_has_modal', 'scl_has_adverb', 'scl_has_nma', 'scl_composition_shift', 'scl_abs_composition_shift', 'scl_shift_zscore', 'scl_shift_percentile', 'scl_shift_strong', 'scl_shift_very_strong', 'scl_composition_type']


## Explode: one row per (movie, keyword)

In [3]:
# Movie columns to carry into the exploded table
MOVIE_META = ['id', 'title', 'quality', 'theatrical', 'release_date',
              'genres', 'original_language', 'popularity', 'vote_count',
              'vote_average', 'imdb_id']

has_kw = movies[movies['keywords'].notna() & (movies['keywords'].str.strip() != '')].copy()
print(f'Movies with keywords: {len(has_kw):,}')

# Explode
has_kw['keyword'] = has_kw['keywords'].str.split(', ')
exploded = (
    has_kw[MOVIE_META + ['keyword']]
    .explode('keyword')
    .reset_index(drop=True)
)
exploded['keyword'] = exploded['keyword'].str.strip()
exploded = exploded[exploded['keyword'] != '']

print(f'Exploded rows: {len(exploded):,}')

# Join keyword lexicon scores
exploded = exploded.merge(kw_lex.drop(columns=['n_movies']), on='keyword', how='left')
print(f'After join: {exploded.shape}')

# Cast boolean columns that become object after left join (NaN introduced for unmatched keywords)
bool_cols = ['emolex_found', 'intensity_found', 'vad_found']
for col in bool_cols:
    if col in exploded.columns:
        exploded[col] = exploded[col].fillna(False).astype(bool)

Movies with keywords: 283,875

Exploded rows: 957,091
After join: (957091, 56)


In [4]:
# Coverage check
scored = exploded['keyword_valence'].notna()
print(f'Keywords with valence score: {scored.sum():,} / {len(exploded):,} ({scored.mean()*100:.1f}%)')
print(f'Valence sources:')
print(exploded['keyword_valence_source'].value_counts().to_string())

exploded.to_csv(DATA / 'tmdb_movie_keywords_exploded.csv', index=False)
print(f'\nSaved → data/tmdb_movie_keywords_exploded.csv  ({exploded.shape[1]} cols)')

Keywords with valence score: 787,335 / 957,091 (82.3%)
Valence sources:
keyword_valence_source
vad         364262
computed    336177
opp          46552
nma          40344



Saved → data/tmdb_movie_keywords_exploded.csv  (56 cols)


## Aggregate: one row per movie

Mean across all scored keywords. Valence is the focus — also aggregate the
8 emotion intensities and 3 VAD dimensions.

In [5]:
EMOTIONS   = ['anger','anticipation','disgust','fear','joy','sadness','surprise','trust']
VAD_DIMS   = ['valence','arousal','dominance']

agg_cols = {
    'keyword_valence':    'movie_valence',
    **{f'vad_{d}':        f'movie_vad_{d}' for d in VAD_DIMS},
    **{f'intensity_{e}':  f'movie_intensity_{e}' for e in EMOTIONS},
    'scl_composition_shift': 'movie_scl_shift',
    'scl_found':          'movie_scl_coverage',
}

agg = (
    exploded
    .groupby('id')[list(agg_cols.keys())]
    .mean()
    .rename(columns=agg_cols)
    .reset_index()
)

# Derived sentiment label from movie_valence
agg['movie_sentiment'] = pd.cut(
    agg['movie_valence'],
    bins=[-1, -0.05, 0.05, 1],
    labels=['negative', 'neutral', 'positive']
)

# Join back to full movie metadata
movie_cols = [c for c in movies.columns if c != 'keywords']
result = movies[movie_cols].merge(agg, on='id', how='left')

# Column order: identity → quality signals → valence focus → full VAD → emotions → SCL → metadata
front = (
    ['id', 'title', 'quality', 'theatrical']
    + ['movie_sentiment', 'movie_valence', 'movie_vad_valence']
    + ['movie_vad_arousal', 'movie_vad_dominance']
    + [f'movie_intensity_{e}' for e in EMOTIONS]
    + ['movie_scl_shift', 'movie_scl_coverage']
)
rest = [c for c in result.columns if c not in front]
result = result[front + rest]

print(f'Movie scores shape: {result.shape}')
has_score = result['movie_valence'].notna()
print(f'Movies with valence score: {has_score.sum():,} ({has_score.mean()*100:.1f}%)')
print(f'\nSentiment breakdown:')
print(result['movie_sentiment'].value_counts().to_string())

result.to_csv(DATA / 'tmdb_movie_scores.csv', index=False)
print(f'\nSaved → data/tmdb_movie_scores.csv  ({result.shape[1]} cols)')

Movie scores shape: (1169756, 40)
Movies with valence score: 263,478 (22.5%)

Sentiment breakdown:
movie_sentiment
positive    171191
negative     69826
neutral      22265



Saved → data/tmdb_movie_scores.csv  (40 cols)


## Valence spot-check

In [6]:
cols = ['title', 'quality', 'movie_sentiment', 'movie_valence', 'movie_vad_valence', 'genres']

real = result[result['quality'].isin(['real_confident', 'real_likely']) & result['movie_valence'].notna()]

print('=== Most positive (real movies) ===')
print(real.nlargest(8, 'movie_valence')[cols].to_string(index=False))

print('\n=== Most negative (real movies) ===')
print(real.nsmallest(8, 'movie_valence')[cols].to_string(index=False))

print('\n=== Biggest SCL composition shift ===')
shifted = result[result['movie_scl_shift'].notna()].nlargest(8, 'movie_scl_shift')
print(shifted[['title', 'quality', 'movie_valence', 'movie_scl_shift', 'genres']].to_string(index=False))

=== Most positive (real movies) ===
                              title        quality movie_sentiment  movie_valence  movie_vad_valence         genres
             Patisserie Coin De Rue real_confident        positive            1.0                1.0          Drama
                    Drums of Tahiti    real_likely        positive            1.0                1.0      Adventure
                       Tiara Tahiti real_confident        positive            1.0                1.0  Comedy, Drama
      Curiosity, Adventure and Love    real_likely        positive            1.0                1.0    Documentary
                           Paradise    real_likely        positive            1.0                1.0 Drama, Fantasy
                   Go with Your Gut    real_likely        positive            1.0                1.0    Documentary
                Gandhi of the Month    real_likely        positive            1.0                1.0          Drama
Kamal Joumblatt, Witness and Martyr 